In [ ]:
##this script is used for building inhouse msms spectra from standard compounds, extract experimental ms spectra from automsms data, and compare the spectra from experiments and the inhouse library for identification
##for QA compdouds,

######################install necessary packages (uncomment if not installed)######################
# !pip install pyopenms
# !pip install tabulate
# !pip install matchms

In [ ]:
#helper functions
import re
import numpy as np
import matchms 
from matchms import calculate_scores
from matchms import Spectrum
from matchms.similarity import CosineGreedy
import pandas as pd
import pyopenms as oms
import tabulate
import os   
import copy
import warnings
# Suppress specific warnings
warnings.filterwarnings("ignore", message=".*")

def preprocess_spectra(filepath, target_mz, ce_level, tolerance=5):
    # Extract spectra with specified collision energy level
    # Filter spectra based on targeted m/z value
    if target_mz is None:
        raise ValueError("Please provide a target m/z value.")
    ce_level = float(ce_level)
    tolerance = float(tolerance)
    target_mz = float(target_mz)
    spectra = oms.MSExperiment()
    oms.MzMLFile().load(filepath, spectra)
    ms2_spec = spectra.getSpectra()
    ms2_query = oms.MSExperiment()

    for s in ms2_spec:
        if s.getMSLevel() == 1:
            continue
        precursors = s.getPrecursors()
        mslevel = s.getMSLevel()
        collision_energy = precursors[0].getMetaValue("collision energy")
        ms_error = ppm_error(precursors[0].getMZ(), target_mz)
        if ms_error < tolerance and mslevel == 2 and collision_energy == ce_level:
            ms2_query.addSpectrum(s)
        else :
            continue

    ms2beforemerg = [s for s in ms2_query.getSpectra() if s.getMSLevel() == 2]
    print(f'Number of MS2 spectra before merge: {len(ms2beforemerg)}')

    if len(ms2beforemerg) == 0:
        print(f"No spectra found with specified parameters for {ce_level}.")
        return None
    
    elif len(ms2beforemerg) >= 1:
        #merget the ms spectra
        merger = oms.SpectraMerger()
        param = merger.getParameters()
        param.setValue("precursor_method:rt_tolerance", 5.0)
        param.setValue("precursor_method:mz_tolerance", 1e-3)
        merger.setParameters(param)
        merger.mergeSpectraPrecursors(ms2_query)
        ms2aftermerg = [s for s in ms2_query.getSpectra() if s.getMSLevel() == 2]
        # print(f'Number of MS2 spectra after merge: {len(ms2aftermerg)}')
        return ms2aftermerg

#normalize spectrum intensities to the maximum intensity for all peaks and return m/z array and intensity array
def normalize_spectrum(spectrum):
    try:
        mz, intensity = spectrum.get_peaks()
    except AttributeError: 
        if isinstance(spectrum, list) and all(isinstance(i, tuple) for i in spectrum):
            mz, intensity = zip(*spectrum)
            mz = np.array(mz)
            intensity = np.array(intensity)
        else:
            raise ValueError("Spectrum format is not recognized.")
    except:
        if isinstance(spectrum, tuple) and len(spectrum) == 2:
            mz = np.array(spectrum[0])
            intensity = np.array(spectrum[1])
        else:
            raise ValueError("Spectrum format is not recognized.")

    # Normalize intensities to the maximum intensity
    if len(intensity) == 0:
        raise ValueError("Intensity array is empty. Cannot normalize an empty spectrum.")
    
    max_intensity = np.max(intensity)
    if max_intensity == 0:
        raise ValueError("Maximum intensity is zero. Cannot normalize spectrum.")
    int_res = intensity / max_intensity * 100  # Scale to percentage
    mz_res = mz.copy()  # Copy m/z values to preserve original order 

    # Assert that the length of mz and intensity are the same
    if len(mz_res) != len(int_res):
        raise ValueError("The length of mz and intensity arrays do not match after filtering.")
    
    # Sort mz values in ascending order and reorder intensities correspondingly.
    sorted_indices = np.argsort(mz_res)
    mz_res = mz_res[sorted_indices]
    int_res = int_res[sorted_indices]
    return mz_res, int_res

def get_spectrum_by_key(spectra, key):
    """
    Retrieves all spectrum information by InChIKey or SMILES.
    Args:
        spectra (dict): Parsed spectra data.
        key (str): The InChIKey or SMILES to search for.

    Returns:
        dict: A dictionary where keys are collision energies and values are spectra.
    """
    if key not in spectra:
        return {}

    grouped_spectra = {}
    for spec in spectra[key]:
        collision_energy = spec["metadata"].get("Collision_energy", spec["metadata"].get("COLLISIONENERGY", "Unknown"))
        if collision_energy not in grouped_spectra:
            grouped_spectra[collision_energy] = []
        grouped_spectra[collision_energy].append(spec["spectrum"])
    return grouped_spectra

# targeted_mzml_to_msp_merge_with_SpectraMerger_POS.py
from __future__ import annotations
import os, re
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
from pyopenms import MSExperiment, MzMLFile, MSSpectrum, SpectraMerger

# ----------------------- helpers -----------------------

def ppm_error(obs_mz: float, ref_mz: float) -> float:
    return (obs_mz - ref_mz) / ref_mz * 1e6

def within_ppm(obs_mz: float, ref_mz: float, ppm: float) -> bool:
    return abs(ppm_error(obs_mz, ref_mz)) <= ppm

def normalize_and_filter(
    mzs: np.ndarray,
    ints: np.ndarray,
    min_rel_int: float = 0.5,
    top_n: Optional[int] = 400
) -> Tuple[np.ndarray, np.ndarray]:
    """Base-peak normalize to 100, drop < min_rel_int %, keep top N peaks."""
    if mzs.size == 0:
        return mzs, ints
    # guard against tiny negatives from vendor processing
    ints = np.clip(ints, 0, None)
    mx = float(ints.max()) if ints.size else 0.0
    if mx > 0:
        ints = (ints / mx) * 100.0
    if min_rel_int > 0:
        mask = ints >= min_rel_int
        mzs, ints = mzs[mask], ints[mask]
    if top_n is not None and ints.size > top_n:
        idx = np.argsort(ints)[-top_n:]
        idx.sort()
        mzs, ints = mzs[idx], ints[idx]
    return mzs, ints

def _sanitize_name(s: str) -> str:
    return re.sub(r"\s+", "_", s.strip())[:200]

def write_msp_entry(
    fh,
    name: str,
    precursor_mz: float,
    charge: Optional[int],
    ion_mode: str,
    collision_energy: Optional[float],
    inchi_key: Optional[str],
    rt_min: Optional[float],
    adduct: Optional[str],
    peaks_mz: np.ndarray,
    peaks_int: np.ndarray,
    comment: Optional[str] = None,
):
    name = _sanitize_name(name)
    fh.write(f"Name: {name}\n")
    fh.write(f"PrecursorMZ: {precursor_mz:.6f}\n")
    if charge is not None and charge != 0:
        fh.write(f"Charge: {charge}\n")
    # you prefer explicit adduct string here:
    if isinstance(adduct, str) and adduct.strip():
        fh.write(f"Precursor_type: {adduct.strip()}\n")
    fh.write(f"Ion_mode: {ion_mode}\n")
    if collision_energy is not None:
        fh.write(f"Collision_energy: {collision_energy}\n")
    if rt_min is not None:
        fh.write(f"Retention_time: {rt_min:.3f}\n")
    if isinstance(inchi_key, str) and inchi_key.strip():
        fh.write(f"INCHIKEY: {inchi_key.strip()}\n")
    if comment:
        fh.write(f"Comment: {comment}\n")

    fh.write(f"Num Peaks: {len(peaks_mz)}\n")
    for m, i in zip(peaks_mz, peaks_int):
        fh.write(f"{m:.5f}\t{i:.1f}\n")
    fh.write("\n")

# ----------------- select + merge via SpectraMerger -----------------

# extra CE keys that show up in some vendor conversions
_CE_KEYS = ("collision energy", "Collision_energy", "CE", "HCD_energy", "cid_energy", "ms2_ae")

def _read_collision_energy(spec) -> Optional[float]:
    ce = None
    if spec.getPrecursors():
        p = spec.getPrecursors()[0]
        try:
            v = float(p.getCollisionEnergy())
            ce = v if v != 0 else None
        except Exception:
            pass
        if ce is None and p.metaValueExists("collision energy"):
            try:
                ce = float(p.getMetaValue("collision energy"))
            except Exception:
                pass
    if ce is None:
        for k in _CE_KEYS:
            if spec.metaValueExists(k):
                try:
                    ce = float(spec.getMetaValue(k))
                    break
                except Exception:
                    continue
    return ce

def select_ms2_for_target_ce(
    exp: MSExperiment,
    target_precursor_mz: float,
    target_rt_min: Optional[float],
    ce_nominal: float,
    ppm_tol_prec: float = 10.0,
    rt_tolerance_sec: float = .010,
    ce_tolerance: float = 0.6,
) -> MSExperiment:
    """
    Build a tiny MSExperiment of just MS2 scans that match:
    - precursor m/z within ppm
    - RT within ± rt_tolerance_sec (seconds)
    - collision energy within ± ce_tolerance of ce_nominal
    """
    out = MSExperiment()

    for spec in exp.getSpectra():
        if spec.getMSLevel() != 2 or not spec.getPrecursors():
            continue

        prec = spec.getPrecursors()[0]
        pmz = float(prec.getMZ()) if prec.getMZ() > 0 else None
        if pmz is None or not within_ppm(pmz, target_precursor_mz, ppm_tol_prec):
            continue

        # RT filter (exp stores seconds)
        rt_sec = float(spec.getRT()) if spec.getRT() is not None else None
        if target_rt_min is not None and rt_sec is not None:
            if abs((rt_sec / 60.0) - target_rt_min) > (rt_tolerance_sec / 60.0):
                continue

        ce = _read_collision_energy(spec)
        if ce is None or abs(ce - float(ce_nominal)) > ce_tolerance:
            continue

        out.addSpectrum(spec)

    return out

def merge_selected_with_spectramerger(
    ms2_query: MSExperiment,
    mz_tolerance_da: float = 1e-3,
    rt_tolerance_sec: float = 10.0
) -> MSExperiment:
    """
    Use pyOpenMS SpectraMerger.mergeSpectraPrecursors to merge selected spectra.
    If there is <2 MS2 spectra, skip merging and return as-is (prevents
    'Distance matrix ... only contains one element' runtime error).
    """
    # Count MS2 in this tiny experiment
    n_ms2 = sum(1 for s in ms2_query.getSpectra() if s.getMSLevel() == 2)
    if n_ms2 < 2:
        # nothing to merge — just return the input
        return ms2_query

    merger = SpectraMerger()
    params = merger.getParameters()
    params.setValue("precursor_method:mz_tolerance", float(mz_tolerance_da))   # Th (Da)
    params.setValue("precursor_method:rt_tolerance", float(rt_tolerance_sec))  # s
    merger.setParameters(params)

    try:
        merger.mergeSpectraPrecursors(ms2_query)  # in-place
    except RuntimeError as e:
        # If anything odd happens, fall back to unmerged spectra
        print(f"[WARN] SpectraMerger failed ({e}). Using unmerged spectra for this CE.")
        return ms2_query

    return ms2_query


# ---------------------- per‑chemical driver ----------------------

def process_one_chemical(
    mzml_path: str,
    inchikey: Optional[str],
    group_id: str,
    target_precursor_mz: float,
    target_rt_min: Optional[float],
    ce_values: List[float],
    ppm_tol_prec: float = 10.0,
    rt_tolerance_sec: float = 30.0,
    ce_tolerance: float = 0.6,
    mz_tolerance_da: float = 1e-3,
    peak_min_rel_int: float = 0.5,
    peak_top_n: Optional[int] = 400,
) -> Dict[float, Dict]:
    """
    Returns dict CE -> meta+peaks for MSP.
    Uses SpectraMerger to merge replicates per CE for this chemical.
    """
    exp = MSExperiment()
    MzMLFile().load(mzml_path, exp)

    results: Dict[float, Dict] = {}

    for ce_nom in ce_values:
        # 1) select scans for this target & CE
        ms2_query = select_ms2_for_target_ce(
            exp=exp,
            target_precursor_mz=target_precursor_mz,
            target_rt_min=target_rt_min,
            ce_nominal=float(ce_nom),
            ppm_tol_prec=ppm_tol_prec,
            rt_tolerance_sec=rt_tolerance_sec,
            ce_tolerance=ce_tolerance,
        )

        n_before = sum(1 for s in ms2_query.getSpectra() if s.getMSLevel() == 2)
        if n_before == 0:
            continue

        # 2) merge replicates
        ms2_merged = merge_selected_with_spectramerger(
            ms2_query,
            mz_tolerance_da=mz_tolerance_da,
            rt_tolerance_sec=rt_tolerance_sec
        )

        merged_specs = [s for s in ms2_merged.getSpectra() if s.getMSLevel() == 2]
        if not merged_specs:
            continue

        # if multiple, pick the one whose precursor m/z is closest (ppm) to target
        def spec_key(s: MSSpectrum):
            p = s.getPrecursors()[0] if s.getPrecursors() else None
            mz = float(p.getMZ()) if (p and p.getMZ() > 0) else np.nan
            return abs(ppm_error(mz, target_precursor_mz)) if np.isfinite(mz) else 1e12

        spec_best = sorted(merged_specs, key=spec_key)[0]

        # Extract peaks & meta
        mzs, ints = spec_best.get_peaks()
        mzs = np.asarray(mzs, dtype=float)
        ints = np.asarray(ints, dtype=float)
        mzs, ints = normalize_and_filter(mzs, ints, min_rel_int=peak_min_rel_int, top_n=peak_top_n)
        if mzs.size == 0:
            continue

        prec = spec_best.getPrecursors()[0] if spec_best.getPrecursors() else None
        prec_mz = float(prec.getMZ()) if (prec and prec.getMZ() > 0) else target_precursor_mz
        charge = int(prec.getCharge()) if (prec and prec.getCharge() != 0) else 1
        rt_min = spec_best.getRT()/60.0 if spec_best.getRT() is not None else target_rt_min

        results[float(ce_nom)] = {
            "precursor_mz": prec_mz,
            "charge": charge,
            "rt_min": rt_min,
            "peaks": (mzs, ints),
        }

    return results

# ---------------------- main orchestration ----------------------
def build_inhouse_msp_from_targets_with_merger(
    pos_entact_csv: str,
    mzml_dir: str,
    out_msp: str,
    ce_values: List[float] = (10.0, 20.0, 40.0),
    ppm_tol_prec: float = 10.0,
    rt_tolerance_sec: float = 5.0,
    ce_tolerance: float = 0.6,
    mz_tolerance_da: float = 1e-3,    # Th for SpectraMerger precursor grouping
    peak_min_rel_int: float = 0.5,
    peak_top_n: Optional[int] = 400,
    adduct_for_name: str = "[M]+",
):
    df = pd.read_csv(pos_entact_csv)
    needed = ["CAS", "Mass", "MS1_RT", "DTXSID"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in pos_entact: {missing}")

    # robust numeric coercion
    df["Mz"] = pd.to_numeric(df["Mass"], errors="coerce")
    df["RT"] = pd.to_numeric(df["MS1_RT"], errors="coerce")

    n_entries = 0
    with open(out_msp, "w", encoding="utf-8") as fh:
        for idx, row in df.iterrows():
            group_id = str(row["CAS"]).strip()
            inchikey = str(row["DTXSID"]).strip() if pd.notna(row["DTXSID"]) else None
            target_mz = float(row["Mz"]) if pd.notna(row["Mz"]) else None
            if target_mz is None:
                print(f"[WARN] bad M+ at row {idx}, group_id={group_id}; skipping")
                continue
            target_rt = float(row["RT"]) if pd.notna(row["RT"]) else None

            mzml_path = os.path.join(mzml_dir, f"{group_id}.mzML")
            if not os.path.exists(mzml_path):
                print(f"[MISS] mzML not found for group_id={group_id}")
                continue

            ce_map = process_one_chemical(
                mzml_path=mzml_path,
                inchikey=inchikey,
                group_id=group_id,
                target_precursor_mz=target_mz,
                target_rt_min=target_rt,
                ce_values=list(ce_values),
                ppm_tol_prec=ppm_tol_prec,
                rt_tolerance_sec=rt_tolerance_sec,
                ce_tolerance=ce_tolerance,
                mz_tolerance_da=mz_tolerance_da,
                peak_min_rel_int=peak_min_rel_int,
                peak_top_n=peak_top_n,
            )

            if not ce_map:
                print(f"[NO-HIT] {group_id} ({inchikey}) no spectra matched filters.")
                continue

            for ce in sorted(ce_map.keys()):
                meta = ce_map[ce]
                pmz = meta["precursor_mz"]
                ch = meta["charge"]
                rt_min = meta["rt_min"]
                mzs, ints = meta["peaks"]

                name = f"{inchikey or 'Unknown'}|{group_id}|CE{int(ce)}"
                write_msp_entry(
                    fh=fh,
                    name=name,
                    precursor_mz=pmz,
                    charge=ch,
                    ion_mode="Positive",
                    collision_energy=ce,
                    inchi_key=inchikey,
                    rt_min=rt_min,
                    adduct=adduct_for_name,   # your explicit precursor/adduct label
                    peaks_mz=mzs,
                    peaks_int=ints,
                    comment=f"group_id={group_id}; target_mz={target_mz:.5f}; rt_target={target_rt}"
                )
                n_entries += 1

    print(f"[OK] wrote {n_entries} MSP entries → {os.path.abspath(out_msp)}")

# ----------------------- run example -----------------------
#step1 extract the reference spectra based on the compound_list from compound_loc by search file ends with .mzML
#step2 extract the experimental spectrum from dda msms data for each inquiry compound based on the mz_match_list, from experimental files in sample_loc directory
#step3 calculate the similarity score between experimental spectram and the reference spectra based on the search key from reference library.
#step4 save the similarity score for each inquiry compound per row, and record the socre for each collision energy level if there are multiple spectra for the same compound, and record the score for each database if there are multiple databases.
#step5 report the results

#file location
compound_loc = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\03042026"
compound_list = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\03042026\compound_list.csv"
mz_match_list = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\mathced_peaks_summary_forallsamples_pos_QACs_formsmsacquisition_20260225.csv"
library_loc = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\LIB2NIST\QACs_51_STD_and_emerging_A.msp"
sample_loc = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\JY_QA_02032026"


if __name__ == "__main__":
    POS_ENTACT_CSV = compound_list
    MZML_DIR = compound_loc
    OUT_MSP = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\QA_pos_spectra.msp"

    build_inhouse_msp_from_targets_with_merger(
        pos_entact_csv=POS_ENTACT_CSV,
        mzml_dir=MZML_DIR,
        out_msp=OUT_MSP,
        ce_values=(10.0, 20.0, 40.0),
        ppm_tol_prec=10.0,        # precursor match tolerance (ppm)
        rt_tolerance_sec=30.0,     # seconds
        ce_tolerance=0.6,         # CE binning tolerance
        mz_tolerance_da=1e-3,     # Th tolerance for SpectraMerger precursor grouping
        peak_min_rel_int=0.5,     # prune <0.5% after merge
        peak_top_n=400,           # cap peaks
        adduct_for_name="[M]+"
    )


KeyboardInterrupt: 

In [2]:
#iterate each row for Average Mz, Average Rt(min), and DTXSID
import os
import pyopenms
from pyopenms.plotting import mirror_plot_spectrum

#extracting spectra from dda files and make inquiry by the QA spectra list by go through each mzml file
def preprocess_spectra2(filepath, target_mz, target_rt, ce_level, tolerance=5):
    # Extract spectra with specified collision energy level
    # Filter spectra based on targeted m/z value
    if target_mz is None:
        raise ValueError("Please provide a target m/z value.")
    ce_level = float(ce_level)
    tolerance = float(tolerance)
    target_mz = float(target_mz)
    target_rt = float(target_rt)
    spectra = oms.MSExperiment()
    oms.MzMLFile().load(filepath, spectra)
    ms2_spec = spectra.getSpectra()

    ms2_query = oms.MSExperiment()
    for s in ms2_spec:
        if s.getMSLevel() == 1:
            continue
        precursors = s.getPrecursors()
        mslevel = s.getMSLevel()
        collision_energy = precursors[0].getMetaValue("collision energy")
        ms_error = ppm_error(precursors[0].getMZ(), target_mz)
        rt_error = abs(s.getRT()/60.0 - target_rt) 
        if ms_error < tolerance and mslevel == 2 and collision_energy == ce_level and rt_error <=0.5:
            ms2_query.addSpectrum(s)
        else :
            continue

    ms2beforemerg = [s for s in ms2_query.getSpectra() if s.getMSLevel() == 2]
    print(f'Number of MS2 spectra before merge: {len(ms2beforemerg)}')

    if len(ms2beforemerg) == 1:
        return ms2beforemerg

    elif len(ms2beforemerg) == 0:
        print(f"No spectra found with specified parameters for {ce_level}.")
        return None
    
    elif len(ms2beforemerg) > 1:
        #merget the ms spectra
        merger = oms.SpectraMerger()
        param = merger.getParameters()
        param.setValue("precursor_method:rt_tolerance", 5.0)
        param.setValue("precursor_method:mz_tolerance", 1e-3)
        merger.setParameters(param)
        merger.mergeSpectraPrecursors(ms2_query)
        ms2aftermerg = [s for s in ms2_query.getSpectra() if s.getMSLevel() == 2]
        # return ms2_query
        return ms2aftermerg

def normalize_and_filter(spectrum, baseline):
    try:
        mz, intensity = spectrum.get_peaks()
    except AttributeError: 
        # mz, intensity = zip(*[(float(m), float(i)) for m, i in spectrum if isinstance(m, (int, float)) and isinstance(i, (int, float))])
        mz, intensity = zip(*spectrum)
        mz = np.array(mz)
        intensity = np.array(intensity)
    except:
        raise ValueError("Please provide a valid spectrum object.")
    
    intensity = np.round((intensity / max(intensity)) * 100, 1)
    indices = np.where(intensity >= baseline)
    return mz[indices], intensity[indices]


#function to get spectrum
def get_spectrum(spectrum):
    try:
        mz, intensity = spectrum.get_peaks()
    except AttributeError:
        mz, intensity = zip(*spectrum)
        mz = np.array(mz)
        intensity = np.array(intensity)
    return mz, intensity

def parse_msp_file(msp_file_path, polarity):
    """
    Parses the .msp file to extract all spectrum information.
    
    Args:
        msp_file_path (str): Path to the .msp file.

    Returns:
        dict: A dictionary where keys are InChIKey or SMILES and values are lists of spectra data.
    """
    spectra = {}
    if polarity == "positive":
        p_type = "[M]+"
    else:
        raise ValueError("Invalid polarity..")
    
    with open(msp_file_path, 'r', encoding='utf-8') as file:
        content = file.read()
        entries = content.split('Name: ')
        for entry in entries[1:]:
            lines = entry.strip().split('\n')
            metadata = {}
            spectrum_data = []
            key = None

            for line in lines:
                if line.startswith("DTXSID:"):
                    key = line.split(": ")[1].strip()
                elif line.startswith("SMILES:") and key is None:
                    key = line.split(": ")[1].strip()
                # elif line.startswith("Spectrum_type:"):
                #     spectrum_type = line.split(": ")[1].strip()
                elif line.startswith("Precursor_type:"):
                    precursor_type = line.split(": ")[1].strip()
                elif line.startswith("Num Peaks:"):
                    num_peaks = int(line.split(": ")[1].strip())
                    spectrum_data = lines[lines.index(line)+1:lines.index(line)+1+num_peaks]
                elif ": " in line:
                    k, v = line.split(": ", 1)
                    metadata[k.strip()] = v.strip()

            # Retain only spectra with "Spectrum_type: MS2"
            if precursor_type == p_type  and key and spectrum_data:
                if key not in spectra:
                    spectra[key] = []
                spectra[key].append({
                    "metadata": metadata,
                    "spectrum": [(float(mz), float(intensity)) for mz, intensity in (line.split() for line in spectrum_data)]
                })
    return spectra

def get_spectrum_by_key(spectra, key):
    """
    Retrieves all spectrum information by InChIKey or SMILES.
    Args:
        spectra (dict): Parsed spectra data.
        key (str): The InChIKey or SMILES to search for.

    Returns:
        dict: A dictionary where keys are collision energies and values are spectra.
    """
    if key not in spectra:
        return {}

    grouped_spectra = {}
    for spec in spectra[key]:
        collision_energy = spec["metadata"].get("Collision_energy", spec["metadata"].get("COLLISIONENERGY", "Unknown"))
        if collision_energy not in grouped_spectra:
            grouped_spectra[collision_energy] = []
        grouped_spectra[collision_energy].append(spec["spectrum"])
    return grouped_spectra

In [ ]:
import os
import numpy as np
import pandas as pd
import pyopenms as oms
from matchms import Spectrum
from matchms.similarity import CosineGreedy
from concurrent.futures import ProcessPoolExecutor, as_completed

# ----------------------- Helpers -----------------------
def parse_msp_file(msp_file_path, polarity):
    """
    Parses the .msp file to extract all spectrum information.
    """
    spectra = {}
    if polarity == "positive":
        p_type = "[M]+"
    else:
        raise ValueError("Invalid polarity.")
    
    with open(msp_file_path, 'r', encoding='utf-8') as file:
        content = file.read()
        entries = content.split('Name: ')
        for entry in entries[1:]:
            lines = entry.strip().split('\n')
            metadata = {}
            spectrum_data = []
            
            # --- THE FIX: Initialize these before looping through the lines ---
            key = None
            precursor_type = None 

            for line in lines:
                # Capture DTXSID or fallback to INCHIKEY
                if line.startswith("DTXSID:"):
                    key = line.split(": ")[1].strip()
                elif line.startswith("INCHIKEY:") and key is None:
                    key = line.split(": ")[1].strip()
                elif line.startswith("Precursor_type:"):
                    precursor_type = line.split(": ")[1].strip()
                elif line.startswith("Num Peaks:"):
                    num_peaks = int(line.split(": ")[1].strip())
                    start_idx = lines.index(line) + 1
                    spectrum_data = lines[start_idx : start_idx + num_peaks]
                elif ": " in line:
                    k, v = line.split(": ", 1)
                    metadata[k.strip()] = v.strip()

            # Retain only spectra with the correct precursor type
            if precursor_type == p_type and key and spectrum_data:
                if key not in spectra:
                    spectra[key] = []
                spectra[key].append({
                    "metadata": metadata,
                    "spectrum": [(float(mz), float(intensity)) for mz, intensity in (line.split() for line in spectrum_data)]
                })
    return spectra



def ppm_error(obs_mz: float, ref_mz: float) -> float:
    return (obs_mz - ref_mz) / ref_mz * 1e6

# [Insert your normalize_and_filter, parse_msp_file, get_spectrum_by_key here]

def preprocess_spectra_batch(exp_data, target_mz, target_rt, ce_level, tolerance=5):
    """Extracts and merges spectra from a pre-loaded MSExperiment."""
    if target_mz is None:
        return None
    
    ce_level = float(ce_level)
    tolerance = float(tolerance)
    target_mz = float(target_mz)
    target_rt = float(target_rt)
    
    ms2_spec = exp_data.getSpectra()
    ms2_query = oms.MSExperiment()
    
    for s in ms2_spec:
        if s.getMSLevel() == 1:
            continue
        precursors = s.getPrecursors()
        if not precursors:
            continue
            
        try:
            collision_energy = float(precursors[0].getMetaValue("collision energy"))
        except (TypeError, ValueError):
            collision_energy = None
            
        ms_error = abs(ppm_error(precursors[0].getMZ(), target_mz))
        rt_error = abs(s.getRT() / 60.0 - target_rt) 
        
        if ms_error < tolerance and s.getMSLevel() == 2 and collision_energy == ce_level and rt_error <= 0.5:
            ms2_query.addSpectrum(s)

    ms2beforemerg = [s for s in ms2_query.getSpectra() if s.getMSLevel() == 2]

    if len(ms2beforemerg) == 1:
        return ms2beforemerg
    elif len(ms2beforemerg) == 0:
        return None
    else:
        merger = oms.SpectraMerger()
        param = merger.getParameters()
        param.setValue("precursor_method:rt_tolerance", 5.0)
        param.setValue("precursor_method:mz_tolerance", 1e-3)
        merger.setParameters(param)
        merger.mergeSpectraPrecursors(ms2_query)
        return [s for s in ms2_query.getSpectra() if s.getMSLevel() == 2]

# ----------------------- Worker Function -----------------------
def process_single_file(file_name: str, query_dir: str, mzmatch_df: pd.DataFrame, ms2_ref_lib: dict, target_ce: float) -> list:
    """
    Worker function executed by the ProcessPool. 
    Handles loading a single mzML file and matching all targets.
    """
    file_path = os.path.join(query_dir, file_name)
    file_results = []
    ce_key = str(float(target_ce))
    
    # Load the mzML file
    exp_data = oms.MSExperiment()
    oms.MzMLFile().load(file_path, exp_data)
    
    # Iterate through target compounds
    for index, row in mzmatch_df.iterrows():
        targetmz = row['Average Mz']
        targetrt = row['Average Rt(min)']
        dtxsid = str(row['DTXSID']).strip()

        ms2_query = preprocess_spectra_batch(exp_data, targetmz, targetrt, target_ce, tolerance=10)
        ms2_ref_spec_dict = get_spectrum_by_key(ms2_ref_lib, dtxsid)

        score_val = 0.0
        matched_peaks = 0
        status = "Success"

        if not ms2_query:
            status = "No query spectra found"
        elif ce_key not in ms2_ref_spec_dict or len(ms2_ref_spec_dict[ce_key]) == 0:
            status = "No reference spectra found"
        else:
            ref_raw_spectrum = ms2_ref_spec_dict[ce_key][0]
            ref_mz, ref_int = normalize_and_filter(ref_raw_spectrum, baseline=10)
            query_mz, query_int = normalize_and_filter(ms2_query[0], baseline=10)
            
            if len(ref_mz) == 0 or len(query_mz) == 0:
                status = "Peaks lost after baseline filtering"
            else:
                reference = Spectrum(mz=ref_mz, intensities=ref_int)   
                query = Spectrum(mz=query_mz, intensities=query_int)

                cosine_greedy = CosineGreedy(tolerance=0.005, mz_power=1.3, intensity_power=0.53)
                score_dict = cosine_greedy.pair(reference, query)
                
                score_val = float(score_dict['score'])
                matched_peaks = int(score_dict['matches'])

        file_results.append({
            "mzML_File": file_name,
            "DTXSID": dtxsid,
            "Target_m/z": targetmz,
            "Target_RT": targetrt,
            "Cosine_Score": score_val,
            "Matched_Peaks": matched_peaks,
            "Status": status
        })
        
    return file_results

# ----------------------- Main Orchestration -----------------------
if __name__ == "__main__":
    # --- User Provided File Locations ---
    compound_loc = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\03042026"
    compound_list = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\03042026\compound_list.csv"
    mz_match_list = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\mathced_peaks_summary_forallsamples_pos_QACs_formsmsacquisition_20260225.csv"
    library_loc = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\LIB2NIST\QACs_51_STD_and_emerging_A.msp"
    # sample_loc = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\JY_QA_02032026"
    sample_loc = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\Copy of 04072026"

    # 1. Setup paths mapped from your variables
    query_dir = sample_loc  # Directory containing the sample mzMLs
    msp_path = library_loc  # The reference library
    
    # Load the target match list
    print("Loading mz_match_list CSV...")
    mzmatch_df = pd.read_csv(mz_match_list)
    
    query_files = [file for file in os.listdir(query_dir) if file.endswith('.mzML')]
    
    if not query_files:
        print(f"No mzML files found in the directory: {query_dir}")
        exit()

    # 2. Pre-load the static library data into memory ONCE
    print(f"Loading MSP library from {os.path.basename(msp_path)}...")
    ms2_ref = parse_msp_file(msp_path, "positive")
    TARGET_CE = 40.0
    
    all_batch_results = []
    
    # 3. Multiprocessing Execution
    # Determine safe number of workers
    max_cores = max(1, os.cpu_count() - 2) 
    print(f"Starting ProcessPoolExecutor with {max_cores} workers for {len(query_files)} files...")

    with ProcessPoolExecutor(max_workers=max_cores) as executor:
        # Submit all tasks to the pool
        future_to_file = {
            executor.submit(process_single_file, file_name, query_dir, mzmatch_df, ms2_ref, TARGET_CE): file_name 
            for file_name in query_files
        }
        
        # Gather results as they complete
        for future in as_completed(future_to_file):
            file_name = future_to_file[future]
            try:
                # Extend the main results list with the results from this file
                file_results = future.result()
                all_batch_results.extend(file_results)
                print(f"[OK] Finished processing: {file_name}")
            except Exception as exc:
                print(f"[ERROR] {file_name} generated an exception: {exc}")

    # 4. Save aggregated results to CSV
    results_df = pd.DataFrame(all_batch_results)
    output_csv = os.path.join(query_dir, "batch_cosine_results_parallel.csv")
    results_df.to_csv(output_csv, index=False)
    print(f"\nParallel batch processing complete! Results saved to: {output_csv}")

Loading mz_match_list CSV...


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\UCSF_postdoc_topic\\Collaboration\\zheng guomao\\mathced_peaks_summary_forallsamples_pos_QACs_formsmsacquisition_20260225.csv'

##library search for interested chemicals

##the following matching the standard spectrum with the experimental spectrum

In [ ]:
from __future__ import annotations
import os, re
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
from pyopenms import MSExperiment, MzMLFile, MSSpectrum, SpectraMerger
import warnings

warnings.filterwarnings("ignore", message=".*")

# ----------------------- Helpers -----------------------
def ppm_error(obs_mz: float, ref_mz: float) -> float:
    return (obs_mz - ref_mz) / ref_mz * 1e6

def within_ppm(obs_mz: float, ref_mz: float, ppm: float) -> bool:
    return abs(ppm_error(obs_mz, ref_mz)) <= ppm

def normalize_and_filter(mzs: np.ndarray, ints: np.ndarray, min_rel_int: float = 0.5, top_n: Optional[int] = 400) -> Tuple[np.ndarray, np.ndarray]:
    if mzs.size == 0:
        return mzs, ints
    ints = np.clip(ints, 0, None)
    mx = float(ints.max()) if ints.size else 0.0
    if mx > 0:
        ints = (ints / mx) * 100.0
    if min_rel_int > 0:
        mask = ints >= min_rel_int
        mzs, ints = mzs[mask], ints[mask]
    if top_n is not None and ints.size > top_n:
        idx = np.argsort(ints)[-top_n:]
        idx.sort()
        mzs, ints = mzs[idx], ints[idx]
    return mzs, ints

def _sanitize_name(s: str) -> str:
    return re.sub(r"\s+", "_", s.strip())[:200]

def _read_collision_energy(spec) -> Optional[float]:
    _CE_KEYS = ("collision energy", "Collision_energy", "CE", "HCD_energy", "cid_energy", "ms2_ae")
    ce = None
    if spec.getPrecursors():
        p = spec.getPrecursors()[0]
        try:
            v = float(p.getCollisionEnergy())
            ce = v if v != 0 else None
        except Exception:
            pass
        if ce is None and p.metaValueExists("collision energy"):
            try:
                ce = float(p.getMetaValue("collision energy"))
            except Exception:
                pass
    if ce is None:
        for k in _CE_KEYS:
            if spec.metaValueExists(k):
                try:
                    ce = float(spec.getMetaValue(k))
                    break
                except Exception:
                    continue
    return ce

# ----------------------- Formatting & Output -----------------------
def write_msp_entry(
    fh, name: str, precursor_mz: float, charge: Optional[int], ion_mode: str,
    collision_energy: Optional[float], rt_min: Optional[float], adduct: Optional[str],
    peaks_mz: np.ndarray, peaks_int: np.ndarray, inchi_key: Optional[str] = None, 
    dtxsid: Optional[str] = None, comment: Optional[str] = None,
):
    name = _sanitize_name(name)
    fh.write(f"Name: {name}\n")
    fh.write(f"PrecursorMZ: {precursor_mz:.6f}\n")
    if charge is not None and charge != 0:
        fh.write(f"Charge: {charge}\n")
    if isinstance(adduct, str) and adduct.strip():
        fh.write(f"Precursor_type: {adduct.strip()}\n")
    fh.write(f"Ion_mode: {ion_mode}\n")
    if collision_energy is not None:
        fh.write(f"Collision_energy: {collision_energy}\n")
    if rt_min is not None:
        fh.write(f"Retention_time: {rt_min:.3f}\n")
    if isinstance(inchi_key, str) and inchi_key.strip() and inchi_key.lower() != "nan":
        fh.write(f"INCHIKEY: {inchi_key.strip()}\n")
    # -> CRITICAL: Writing DTXSID explicitly so the matcher can find it
    if isinstance(dtxsid, str) and dtxsid.strip() and dtxsid.lower() != "nan":
        fh.write(f"DTXSID: {dtxsid.strip()}\n")
    if comment:
        fh.write(f"Comment: {comment}\n")

    fh.write(f"Num Peaks: {len(peaks_mz)}\n")
    for m, i in zip(peaks_mz, peaks_int):
        fh.write(f"{m:.5f}\t{i:.1f}\n")
    fh.write("\n")

# ----------------------- Extraction & Merging -----------------------
def select_ms2_for_target_ce(exp: MSExperiment, target_precursor_mz: float, target_rt_min: Optional[float], ce_nominal: float, ppm_tol_prec: float = 10.0, rt_tolerance_sec: float = 30.0, ce_tolerance: float = 0.6) -> MSExperiment:
    out = MSExperiment()
    for spec in exp.getSpectra():
        if spec.getMSLevel() != 2 or not spec.getPrecursors():
            continue
        prec = spec.getPrecursors()[0]
        pmz = float(prec.getMZ()) if prec.getMZ() > 0 else None
        if pmz is None or not within_ppm(pmz, target_precursor_mz, ppm_tol_prec):
            continue
        rt_sec = float(spec.getRT()) if spec.getRT() is not None else None
        if target_rt_min is not None and rt_sec is not None:
            if abs((rt_sec / 60.0) - target_rt_min) > (rt_tolerance_sec / 60.0):
                continue
        ce = _read_collision_energy(spec)
        if ce is None or abs(ce - float(ce_nominal)) > ce_tolerance:
            continue
        out.addSpectrum(spec)
    return out

def merge_selected_with_spectramerger(ms2_query: MSExperiment, mz_tolerance_da: float = 1e-3, rt_tolerance_sec: float = 30.0) -> MSExperiment:
    if sum(1 for s in ms2_query.getSpectra() if s.getMSLevel() == 2) < 2:
        return ms2_query
    merger = SpectraMerger()
    params = merger.getParameters()
    params.setValue("precursor_method:mz_tolerance", float(mz_tolerance_da))
    params.setValue("precursor_method:rt_tolerance", float(rt_tolerance_sec))
    merger.setParameters(params)
    try:
        merger.mergeSpectraPrecursors(ms2_query)
    except RuntimeError as e:
        print(f"[WARN] SpectraMerger failed ({e}). Using unmerged spectra.")
    return ms2_query

def process_one_chemical(mzml_path: str, group_id: str, target_precursor_mz: float, target_rt_min: Optional[float], ce_values: List[float], ppm_tol_prec: float = 10.0, rt_tolerance_sec: float = 30.0, ce_tolerance: float = 0.6, mz_tolerance_da: float = 1e-3, peak_min_rel_int: float = 0.5, peak_top_n: Optional[int] = 400) -> Dict[float, Dict]:
    exp = MSExperiment()
    MzMLFile().load(mzml_path, exp)
    results: Dict[float, Dict] = {}

    for ce_nom in ce_values:
        ms2_query = select_ms2_for_target_ce(exp, target_precursor_mz, target_rt_min, float(ce_nom), ppm_tol_prec, rt_tolerance_sec, ce_tolerance)
        if sum(1 for s in ms2_query.getSpectra() if s.getMSLevel() == 2) == 0:
            continue

        ms2_merged = merge_selected_with_spectramerger(ms2_query, mz_tolerance_da, rt_tolerance_sec)
        merged_specs = [s for s in ms2_merged.getSpectra() if s.getMSLevel() == 2]
        if not merged_specs:
            continue

        def spec_key(s: MSSpectrum):
            p = s.getPrecursors()[0] if s.getPrecursors() else None
            mz = float(p.getMZ()) if (p and p.getMZ() > 0) else np.nan
            return abs(ppm_error(mz, target_precursor_mz)) if np.isfinite(mz) else 1e12

        spec_best = sorted(merged_specs, key=spec_key)[0]
        mzs, ints = spec_best.get_peaks()
        mzs, ints = normalize_and_filter(np.asarray(mzs, dtype=float), np.asarray(ints, dtype=float), min_rel_int=peak_min_rel_int, top_n=peak_top_n)
        
        if mzs.size == 0: continue

        prec = spec_best.getPrecursors()[0] if spec_best.getPrecursors() else None
        prec_mz = float(prec.getMZ()) if (prec and prec.getMZ() > 0) else target_precursor_mz
        charge = int(prec.getCharge()) if (prec and prec.getCharge() != 0) else 1
        rt_min = spec_best.getRT()/60.0 if spec_best.getRT() is not None else target_rt_min

        results[float(ce_nom)] = {
            "precursor_mz": prec_mz, "charge": charge, "rt_min": rt_min, "peaks": (mzs, ints),
        }
    return results

def build_inhouse_msp_from_targets(pos_entact_csv: str, mzml_dir: str, out_msp: str, ce_values: List[float], ppm_tol_prec: float, rt_tolerance_sec: float, ce_tolerance: float, mz_tolerance_da: float, peak_min_rel_int: float, peak_top_n: Optional[int], adduct_for_name: str):
    df = pd.read_csv(pos_entact_csv)
    # Ensure DTXSID column exists
    if "DTXSID" not in df.columns:
        raise ValueError("Missing column in compound list: 'DTXSID'")
        
    df["Mz"] = pd.to_numeric(df["Mass"], errors="coerce")
    df["RT"] = pd.to_numeric(df["MS1_RT"], errors="coerce")

    n_entries = 0
    with open(out_msp, "w", encoding="utf-8") as fh:
        for idx, row in df.iterrows():
            group_id = str(row["CAS"]).strip()
            dtxsid = str(row["DTXSID"]).strip() if pd.notna(row["DTXSID"]) else None
            inchikey = str(row.get("INCHIKEY", "")).strip() if pd.notna(row.get("INCHIKEY")) else None
            
            target_mz = float(row["Mz"]) if pd.notna(row["Mz"]) else None
            if target_mz is None: continue
            target_rt = float(row["RT"]) if pd.notna(row["RT"]) else None

            mzml_path = os.path.join(mzml_dir, f"{group_id}.mzML")
            if not os.path.exists(mzml_path): 
                print(f"[MISS] mzML not found for group_id={group_id}")
                continue

            ce_map = process_one_chemical(mzml_path, group_id, target_mz, target_rt, ce_values, ppm_tol_prec, rt_tolerance_sec, ce_tolerance, mz_tolerance_da, peak_min_rel_int, peak_top_n)
            if not ce_map: 
                print(f"[NO-HIT] {group_id} ({dtxsid}) no spectra matched filters.")
                continue

            for ce in sorted(ce_map.keys()):
                meta = ce_map[ce]
                display_id = dtxsid or inchikey or 'Unknown'
                name = f"{display_id}|{group_id}|CE{int(ce)}"
                
                write_msp_entry(
                    fh=fh, name=name, precursor_mz=meta["precursor_mz"], charge=meta["charge"],
                    ion_mode="Positive", collision_energy=ce, rt_min=meta["rt_min"],
                    adduct=adduct_for_name, peaks_mz=meta["peaks"][0], peaks_int=meta["peaks"][1],
                    dtxsid=dtxsid, inchi_key=inchikey, comment=f"group_id={group_id}; target_mz={target_mz:.5f}; rt_target={target_rt}"
                )
                n_entries += 1
    print(f"[OK] wrote {n_entries} MSP entries → {os.path.abspath(out_msp)}")

if __name__ == "__main__":
    # --- Exact Paths ---
    compound_loc = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\03042026"
    compound_list = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\03042026\compound_list.csv"
    
    # Save the custom built library in the main project folder
    out_msp_file = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\QA_pos_spectra_custom.msp"

    print(f"Building custom MSP library from {compound_loc}...")
    
    build_inhouse_msp_from_targets(
        pos_entact_csv=compound_list, 
        mzml_dir=compound_loc, 
        out_msp=out_msp_file,
        ce_values=[10.0, 20.0, 40.0], 
        ppm_tol_prec=10.0, 
        rt_tolerance_sec=30.0,
        ce_tolerance=0.6, 
        mz_tolerance_da=1e-3, 
        peak_min_rel_int=0.5, 
        peak_top_n=400, 
        adduct_for_name="[M]+"
    )

Building custom MSP library from D:\UCSF_postdoc_topic\Collaboration\zheng guomao\03042026...
[MISS] mzML not found for group_id=67-48-1_MS1_pos
[MISS] mzML not found for group_id=112-02-7_MS1_pos
[MISS] mzML not found for group_id=139-07-1_MS1_pos
[MISS] mzML not found for group_id=122-18-9_MS1_pos
[OK] wrote 21 MSP entries → D:\UCSF_postdoc_topic\Collaboration\zheng guomao\QA_pos_spectra_custom.msp


In [ ]:
def parse_msp_file(msp_file_path, polarity):
    spectra = {}
    p_type = "[M]+" if polarity == "positive" else None
    if not p_type: raise ValueError("Invalid polarity.")
    
    with open(msp_file_path, 'r', encoding='utf-8') as file:
        content = file.read()
        entries = content.split('Name: ')
        for entry in entries[1:]:
            lines = entry.strip().split('\n')
            metadata, spectrum_data = {}, []
            key, precursor_type = None, None 

            for line in lines:
                if line.startswith("DTXSID:"): key = line.split(": ")[1].strip()
                elif line.startswith("INCHIKEY:") and key is None: key = line.split(": ")[1].strip()
                elif line.startswith("Precursor_type:"): precursor_type = line.split(": ")[1].strip()
                elif line.startswith("Num Peaks:"):
                    num_peaks = int(line.split(": ")[1].strip())
                    start_idx = lines.index(line) + 1
                    spectrum_data = lines[start_idx : start_idx + num_peaks]
                elif ": " in line:
                    k, v = line.split(": ", 1)
                    metadata[k.strip()] = v.strip()

            if (precursor_type == p_type or precursor_type is None) and key and spectrum_data:
                if key not in spectra: spectra[key] = []
                spectra[key].append({
                    "metadata": metadata,
                    "spectrum": [(float(mz), float(intensity)) for mz, intensity in (line.split() for line in spectrum_data)]
                })
    return spectra


def preprocess_spectra_batch(exp_data, target_mz, target_rt, ce_level, tolerance=5):
    if target_mz is None: return None
    
    ms2_spec = exp_data.getSpectra()
    ms2_query = oms.MSExperiment()
    
    for s in ms2_spec:
        if s.getMSLevel() == 1 or not s.getPrecursors(): continue
        try: collision_energy = float(s.getPrecursors()[0].getMetaValue("collision energy"))
        except (TypeError, ValueError): collision_energy = None
            
        ms_error = abs(ppm_error(s.getPrecursors()[0].getMZ(), float(target_mz)))
        rt_error = abs(s.getRT() / 60.0 - float(target_rt)) 
        
        if ms_error < float(tolerance) and s.getMSLevel() == 2 and collision_energy == float(ce_level) and rt_error <= 0.6:
            ms2_query.addSpectrum(s)

    ms2beforemerg = [s for s in ms2_query.getSpectra() if s.getMSLevel() == 2]
    if len(ms2beforemerg) == 1: return ms2beforemerg
    elif len(ms2beforemerg) == 0: return None
    else:
        merger = oms.SpectraMerger()
        param = merger.getParameters()
        param.setValue("precursor_method:rt_tolerance", 5.0)
        param.setValue("precursor_method:mz_tolerance", 1e-3)
        merger.setParameters(param)
        merger.mergeSpectraPrecursors(ms2_query)
        return [s for s in ms2_query.getSpectra() if s.getMSLevel() == 2]

def normalize_and_filter(spectrum, baseline):
    try:
        mz, intensity = spectrum.get_peaks()
    except AttributeError:
        mz, intensity = zip(*spectrum)
        mz = np.array(mz)
        intensity = np.array(intensity)
    intensity = np.round((intensity / max(intensity)) * 100, 1)
    indices = np.where(intensity >= baseline)
    return mz[indices], intensity[indices]

def get_spectrum_by_key(spectra, key):
    """
    Retrieves all spectrum information by InChIKey or SMILES.
    Args:
        spectra (dict): Parsed spectra data.
        key (str): The InChIKey or SMILES to search for.

    Returns:
        dict: A dictionary where keys are collision energies and values are spectra.
    """
    if key not in spectra:
        return {}

    grouped_spectra = {}
    for spec in spectra[key]:
        collision_energy = spec["metadata"].get("Collision_energy", spec["metadata"].get("COLLISIONENERGY", "Unknown"))
        if collision_energy not in grouped_spectra:
            grouped_spectra[collision_energy] = []
        grouped_spectra[collision_energy].append(spec["spectrum"])
    return grouped_spectra

# # ----------------------- Main Orchestration -----------------------
# if __name__ == "__main__":
#     # --- Exact Paths ---
#     # mz_match_list = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\mathced_peaks_summary_forallsamples_pos_QACs_formsmsacquisition_20260225.csv"
#     mz_match_list = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\mathced_peaks_summary_forallsamples_pos_QACs_formsmsacquisition_20260309.csv" 
#     # query_dir = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\JY_QA_02032026"
#     query_dir = r'D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\samples_automsms_positive'
#     custom_library_loc = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\QA_pos_spectra_custom.msp"
#     # custom_library_loc = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\QACs_51_STD_and_emerging_B.msp"

#     # Create a directory to save the plots
#     plot_dir = os.path.join(query_dir, "Mirror_Plots")
#     os.makedirs(plot_dir, exist_ok=True)

#     print("Loading mz_match_list CSV...")
#     mzmatch_df = pd.read_csv(mz_match_list)
#     query_files = [file for file in os.listdir(query_dir) if file.endswith('.mzML')]
    
#     if not query_files:
#         print(f"No mzML files found in {query_dir}")
#         exit()

#     print(f"Loading custom MSP library from {os.path.basename(custom_library_loc)}...")
#     ms2_ref_lib = parse_msp_file(custom_library_loc, "positive")
#     TARGET_CE = 20.0
#     ce_key = str(float(TARGET_CE))

#     all_batch_results = []
    
#     # 🔴 Loop through every mzML file sequentially
#     for file_name in query_files:
#         file_path = os.path.join(query_dir, file_name)
#         file_prefix = os.path.splitext(file_name)[0]  # Get name without .mzML
        
#         print(f"\n--- Processing: {file_name} ---")
        
#         # Load the mzML file once per iteration
#         exp_data = oms.MSExperiment()
#         try:
#             oms.MzMLFile().load(file_path, exp_data)
#         except Exception as e:
#             print(f"[ERROR] Failed to load {file_name}: {e}")
#             continue
        
#         for index, row in mzmatch_df.iterrows():
#             targetmz = row['Average Mz']
#             targetrt = row['Average Rt(min)']
#             dtxsid = str(row['DTXSID']).strip()

#             ms2_query = preprocess_spectra_batch(exp_data, targetmz, targetrt, TARGET_CE, tolerance=10)
#             ms2_ref_spec_dict = get_spectrum_by_key(ms2_ref_lib, dtxsid)

#             score_val, matched_peaks, status = 0.0, 0, "Success"

#             if not ms2_query: 
#                 status = "No query spectra found"
#             elif ce_key not in ms2_ref_spec_dict or not ms2_ref_spec_dict[ce_key]: 
#                 status = "No reference spectra found in library"
#             else:
#                 ref_mz, ref_int = normalize_and_filter(ms2_ref_spec_dict[ce_key][0], baseline=10)
#                 query_mz, query_int = normalize_and_filter(ms2_query[0], baseline=10)
                
#                 if len(ref_mz) == 0 or len(query_mz) == 0: 
#                     status = "Peaks lost after baseline filtering"
#                 else:
#                     reference = Spectrum(mz=ref_mz, intensities=ref_int)   
#                     query = Spectrum(mz=query_mz, intensities=query_int)
#                     score_dict = CosineGreedy(tolerance=0.005, mz_power=1.3, intensity_power=0.53).pair(reference, query)
#                     score_val = float(score_dict['score'])
#                     matched_peaks = int(score_dict['matches'])

#                     # Plotting Logic
#                     if score_val > 0.0:
#                         plt.figure(figsize=(10, 6))
#                         reference.plot_against(query)
#                         plt.title(f"Mirror Plot: {dtxsid}\nFile: {file_name}\nCosine Score: {score_val:.3f} | Matched Peaks: {matched_peaks}")
                        
#                         plot_filename = f"{file_prefix}_{dtxsid}_score_{score_val:.2f}.pdf"
#                         plt.savefig(os.path.join(plot_dir, plot_filename), bbox_inches="tight")
#                         plt.close()

#             all_batch_results.append({
#                 "mzML_File": file_name, 
#                 "DTXSID": dtxsid, 
#                 "Target_m/z": targetmz,
#                 "Target_RT": targetrt, 
#                 "Cosine_Score": score_val, 
#                 "Matched_Peaks": matched_peaks, 
#                 "Status": status
#             })
            
#             if status == "Success":
#                 print(f"  [MATCH] {dtxsid} | Score: {score_val:.3f} | Peaks: {matched_peaks}")

#     # Output final CSV
#     # combine the query_dir with a subfolder for the target CE to save the results in an organized manner
#     output_dir = os.path.join(query_dir, str(TARGET_CE),'Lib_B')
#     os.makedirs(output_dir, exist_ok=True)
#     output_csv = os.path.join(output_dir, "sequential_batch_cosine_results.csv")
#     if all_batch_results:
#         pd.DataFrame(all_batch_results).to_csv(output_csv, index=False)
#         print(f"\nSequential batch processing complete! Results saved to: {output_csv}")
#         print(f"Mirror plots saved in: {plot_dir}")
#     else:
#         print("\nNo results generated.")

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyopenms as oms
import matchms
from matchms import Spectrum
from matchms.similarity import CosineGreedy

def ppm_error(mz_obs, mz_ref):
    return (mz_obs - mz_ref) / mz_ref * 1e6

def normalize_and_filter(spectrum, baseline):
    try:
        mz, intensity = spectrum.get_peaks()
    except AttributeError:
        mz, intensity = zip(*spectrum)
        mz = np.array(mz)
        intensity = np.array(intensity)
    intensity = np.round((intensity / max(intensity)) * 100, 1)
    indices = np.where(intensity >= baseline)
    return mz[indices], intensity[indices]

def parse_msp_file(msp_file_path, polarity):
    spectra = {}
    p_type = "[M]+" if polarity == "positive" else None
    if not p_type: raise ValueError("Invalid polarity.")
    
    with open(msp_file_path, 'r', encoding='utf-8') as file:
        content = file.read()
        entries = content.split('Name: ')
        for entry in entries[1:]:
            lines = entry.strip().split('\n')
            metadata, spectrum_data = {}, []
            key, precursor_type = None, None 

            for i, line in enumerate(lines):
                if line.startswith("DTXSID:"): 
                    key = line.split(": ")[1].strip()
                elif line.startswith("INCHIKEY:") and key is None: 
                    key = line.split(": ")[1].strip()
                elif line.startswith("Precursor_type:"): 
                    precursor_type = line.split(": ")[1].strip()
                # 100% robust, case-insensitive check for peaks
                elif line.lower().startswith("num peaks:"):
                    try:
                        num_peaks = int(line.split(":")[1].strip())
                        start_idx = i + 1
                        spectrum_data = lines[start_idx : start_idx + num_peaks]
                    except:
                        pass
                elif ": " in line:
                    k, v = line.split(": ", 1)
                    metadata[k.strip()] = v.strip()

            if (precursor_type == p_type or precursor_type is None) and key and spectrum_data:
                if key not in spectra: 
                    spectra[key] = []
                try:
                    spectra[key].append({
                        "metadata": metadata,
                        # Filters out empty lines before trying to parse the numbers
                        "spectrum": [(float(mz), float(intensity)) for mz, intensity in (line.split() for line in spectrum_data if line.strip())]
                    })
                except Exception:
                    pass
    return spectra

def preprocess_spectra_batch_v1_merge(exp_data, target_mz, target_rt, ce_level, tolerance=5, fragment_mz_tol=0.01):
    """V1: Merges multiple MS2 scans and sums peak intensities within fragment_mz_tol."""
    if target_mz is None: return None
    
    ms2_spec = exp_data.getSpectra()
    ms2_query = oms.MSExperiment()
    
    for s in ms2_spec:
        if s.getMSLevel() == 1 or not s.getPrecursors(): continue
        try: collision_energy = float(s.getPrecursors()[0].getMetaValue("collision energy"))
        except (TypeError, ValueError): collision_energy = None
            
        ms_error = abs(ppm_error(s.getPrecursors()[0].getMZ(), float(target_mz)))
        rt_error = abs(s.getRT() / 60.0 - float(target_rt)) 
        
        if ms_error < float(tolerance) and s.getMSLevel() == 2 and collision_energy == float(ce_level) and rt_error <= 0.6:
            ms2_query.addSpectrum(s)

    ms2beforemerg = [s for s in ms2_query.getSpectra() if s.getMSLevel() == 2]
    if len(ms2beforemerg) == 1: 
        return ms2beforemerg
    elif len(ms2beforemerg) == 0: 
        return None
    else:
        merger = oms.SpectraMerger()
        param = merger.getParameters()
        param.setValue("precursor_method:rt_tolerance", 5.0)
        param.setValue("precursor_method:mz_tolerance", 1e-3)
        # CRITICAL ADDITION: Binning width for the MS2 product ions
        param.setValue("mz_binning_width", float(fragment_mz_tol)) 
        param.setValue("mz_binning_width_unit", "Da") 
        merger.setParameters(param)
        
        merger.mergeSpectraPrecursors(ms2_query)
        return [s for s in ms2_query.getSpectra() if s.getMSLevel() == 2]
    
def preprocess_spectra_batch_v2_average(exp_data, target_mz, target_rt, ce_level, tolerance=5, fragment_mz_tol=0.01):
    """V2: Merges multiple MS2 scans and averages peak intensities."""
    if target_mz is None: return None
    
    ms2_spec = exp_data.getSpectra()
    ms2_query = oms.MSExperiment()
    
    for s in ms2_spec:
        if s.getMSLevel() == 1 or not s.getPrecursors(): continue
        try: collision_energy = float(s.getPrecursors()[0].getMetaValue("collision energy"))
        except (TypeError, ValueError): collision_energy = None
            
        ms_error = abs(ppm_error(s.getPrecursors()[0].getMZ(), float(target_mz)))
        rt_error = abs(s.getRT() / 60.0 - float(target_rt)) 
        
        if ms_error < float(tolerance) and s.getMSLevel() == 2 and collision_energy == float(ce_level) and rt_error <= 0.6:
            ms2_query.addSpectrum(s)

    ms2beforemerg = [s for s in ms2_query.getSpectra() if s.getMSLevel() == 2]
    num_spectra = len(ms2beforemerg)
    
    if num_spectra == 1: 
        return ms2beforemerg
    elif num_spectra == 0: 
        return None
    else:
        merger = oms.SpectraMerger()
        param = merger.getParameters()
        param.setValue("precursor_method:rt_tolerance", 5.0)
        param.setValue("precursor_method:mz_tolerance", 1e-3)
        param.setValue("mz_binning_width", float(fragment_mz_tol)) 
        param.setValue("mz_binning_width_unit", "Da") 
        merger.setParameters(param)
        
        merger.mergeSpectraPrecursors(ms2_query)
        merged_spectra = [s for s in ms2_query.getSpectra() if s.getMSLevel() == 2]
        
        # Manually average the intensities by dividing by the number of grouped scans
        if merged_spectra:
            avg_spectrum = merged_spectra[0]
            mz_array, int_array = avg_spectrum.get_peaks()
            avg_spectrum.set_peaks((mz_array, int_array / num_spectra))
            return [avg_spectrum]
        return []

def preprocess_spectra_batch_v3_middle(exp_data, target_mz, target_rt, ce_level, tolerance=5):
    """V3: Selects only the middle MS2 scan sequentially to represent the spectra."""
    if target_mz is None: return None
    
    ms2_spec = exp_data.getSpectra()
    valid_scans = []
    
    for s in ms2_spec:
        if s.getMSLevel() == 1 or not s.getPrecursors(): continue
        try: collision_energy = float(s.getPrecursors()[0].getMetaValue("collision energy"))
        except (TypeError, ValueError): collision_energy = None
            
        ms_error = abs(ppm_error(s.getPrecursors()[0].getMZ(), float(target_mz)))
        rt_error = abs(s.getRT() / 60.0 - float(target_rt)) 
        
        if ms_error < float(tolerance) and s.getMSLevel() == 2 and collision_energy == float(ce_level) and rt_error <= 0.6:
            valid_scans.append(s)

    if len(valid_scans) == 0:
        return None
    elif len(valid_scans) == 1:
        return valid_scans
    else:
        # Sort by RT to ensure they are sequential, then pick the exact middle index
        valid_scans.sort(key=lambda x: x.getRT())
        middle_index = len(valid_scans) // 2
        return [valid_scans[middle_index]]


def get_spectrum_by_key(spectra, key):
    """Retrieves all spectrum information by searching for the DTXSID (Case-Insensitive)."""
    if not key or pd.isna(key):
        return {}

    target_key = str(key).strip().upper()
    matched_spectra = None
    
    for k in spectra.keys():
        if str(k).strip().upper() == target_key:
            matched_spectra = spectra[k]
            break
            
    if not matched_spectra:
        return {}

    grouped_spectra = {}
    for spec in matched_spectra:
        meta_upper = {k.upper(): v for k, v in spec["metadata"].items()}
        ce_raw = meta_upper.get("COLLISION_ENERGY", meta_upper.get("COLLISIONENERGY", "Unknown"))
        
        try:
            ce = str(float(ce_raw))
        except ValueError:
            ce = ce_raw
            
        if ce not in grouped_spectra:
            grouped_spectra[ce] = []
        grouped_spectra[ce].append(spec["spectrum"])
        
    return grouped_spectra

# ----------------------- Main Orchestration -----------------------
if __name__ == "__main__":
    # --- Exact Paths ---
    mz_match_list = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\mathced_peaks_summary_forallsamples_pos_QACs_formsmsacquisition_20260309.csv" 
    query_dir = r'D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\samples_automsms_positive'
    # query_dir = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\JY_QA_02032026"
    # query_dir = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\Copy of JY_03132026"
    # query_dir = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\Copy of 04072026"
    

    # FIX: Pointing to Library B which contains your missing chemical DTXSID2040787!
    # custom_library_loc = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\QA_pos_spectra_custom.msp"
    custom_library_loc = r"D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\QACs_51_STD_and_emerging_B.msp"

    plot_dir = os.path.join(query_dir, "Mirror_Plots")
    os.makedirs(plot_dir, exist_ok=True)

    print("Loading mz_match_list CSV...")
    mzmatch_df = pd.read_csv(mz_match_list)
    query_files = [file for file in os.listdir(query_dir) if file.endswith('.mzML')]
    
    if not query_files:
        print(f"No mzML files found in {query_dir}")
        exit()

    print(f"Loading custom MSP library from {os.path.basename(custom_library_loc)}...")
    ms2_ref_lib = parse_msp_file(custom_library_loc, "positive")
    # TARGET_CE = 10.0
    # TARGET_CE = 20.0
    TARGET_CE = 40.0
    ce_key = str(float(TARGET_CE))

    all_batch_results = []
    
    # Loop through every mzML file sequentially
    for file_name in query_files:
        file_path = os.path.join(query_dir, file_name)
        file_prefix = os.path.splitext(file_name)[0]
        
        print(f"\n--- Processing: {file_name} ---")
        
        exp_data = oms.MSExperiment()
        try:
            oms.MzMLFile().load(file_path, exp_data)
        except Exception as e:
            print(f"[ERROR] Failed to load {file_name}: {e}")
            continue
        
        for index, row in mzmatch_df.iterrows():
            targetmz = row['Average Mz']
            targetrt = row['Average Rt(min)']
            dtxsid = str(row['DTXSID']).strip()

            ms2_query = preprocess_spectra_batch_v1_merge(exp_data, targetmz, targetrt, TARGET_CE, tolerance=10)
            ms2_ref_spec_dict = get_spectrum_by_key(ms2_ref_lib, dtxsid)
            score_val, matched_peaks, status = 0.0, 0, "Success"
            line_width = 4.0
            plot_mzlabels = 0

            if not ms2_query: 
                status = "No query spectra found"
            elif ce_key not in ms2_ref_spec_dict or not ms2_ref_spec_dict[ce_key]: 
                status = "No reference spectra found in library"
            else:
                ref_mz, ref_int = normalize_and_filter(ms2_ref_spec_dict[ce_key][0], baseline=10)
                query_mz, query_int = normalize_and_filter(ms2_query[0], baseline=10)
                
                if len(ref_mz) == 0 or len(query_mz) == 0: 
                    status = "Peaks lost after baseline filtering"
                else:
                    reference = Spectrum(mz=ref_mz, intensities=ref_int)   
                    query = Spectrum(mz=query_mz, intensities=query_int)
                    score_dict = CosineGreedy(tolerance=0.005, mz_power=1.3, intensity_power=0.53).pair(reference, query)
                    score_val = float(score_dict['score'])
                    matched_peaks = int(score_dict['matches'])

                    if score_val > 0.7 and matched_peaks >= 3.0:
                        fig, ax = plt.subplots(figsize=(10, 6))
                        
                        ax.stem(ref_mz, ref_int, linefmt='b-', markerfmt=' ', basefmt='k-', label='Reference (Library)')
                        markerline, stemlines, baseline = ax.stem(ref_mz, ref_int, linefmt='b-', markerfmt=' ', basefmt='k-')
                        plt.setp(stemlines, linewidth=line_width)
                        if plot_mzlabels == 1:
                            for m, i in zip(ref_mz, ref_int):
                                if i > 10.0:
                                    ax.text(m+0.6, i-12, f"{m:.4f}", ha='center', va='bottom', fontsize=9, color='blue', rotation=90)
                                    
                        ax.stem(query_mz, -query_int, linefmt='r-', markerfmt=' ', basefmt='k-', label='Query (Sample)')
                        markerline, stemlines, baseline = ax.stem(query_mz, -query_int, linefmt='r-', markerfmt=' ', basefmt='k-')
                        plt.setp(stemlines, linewidth=line_width)
                        if plot_mzlabels == 1:
                            for m, i in zip(query_mz, query_int):
                                if i > 10.0:  
                                    ax.text(m+0.6, -i+12, f"{m:.4f}", ha='center', va='top', fontsize=9, color='red', rotation=90)

                        ax.axhline(0, color='black', linewidth=2)
                        ax.set_ylim(-115, 115)
                        ax.set_xlabel("m/z", fontweight='bold')
                        ax.set_ylabel("Relative Intensity (%)", fontweight='bold')
                        ax.set_title(f"Mirror Plot: {dtxsid}\nFile: {file_name} | CE:{TARGET_CE}\nCosine Score: {score_val:.3f} | Matched Peaks: {matched_peaks}")
                        # ax.legend(loc='upper right')
                        ax.tick_params(axis='both', which='both', width=line_width, length=7, labelsize=14)
                        ax.spines['top'].set_visible(False)
                        ax.spines['right'].set_visible(False)
                        ax.spines['left'].set_linewidth(line_width)
                        ax.spines['bottom'].set_linewidth(line_width)

                        plot_filename = f"{file_prefix}_{dtxsid}_score_{score_val:.2f}.png"
                        output_dir = os.path.join(query_dir, str(TARGET_CE), 'spectra_plots')
                        os.makedirs(output_dir, exist_ok=True)
                        plt.savefig(os.path.join(output_dir, plot_filename), bbox_inches="tight", dpi=300, format='png')
                        plt.close(fig)

            all_batch_results.append({
                "mzML_File": file_name, 
                "DTXSID": dtxsid, 
                "Target_m/z": targetmz,
                "Target_RT": targetrt, 
                "Cosine_Score": score_val, 
                "Matched_Peaks": matched_peaks, 
                "Status": status
            })
            
            if status == "Success":
                print(f"  [MATCH] {dtxsid} | Score: {score_val:.3f} | Peaks: {matched_peaks}")

    output_dir = os.path.join(query_dir, str(TARGET_CE), 'Lib_B')
    os.makedirs(output_dir, exist_ok=True)
    output_csv = os.path.join(output_dir, "sequential_batch_cosine_results.csv")
    
    if all_batch_results:
        pd.DataFrame(all_batch_results).to_csv(output_csv, index=False)
        print(f"\nSequential batch processing complete! Results saved to: {output_csv}")
        print(f"Mirror plots saved in: {plot_dir}")
    else:
        print("\nNo results generated.")

Loading mz_match_list CSV...
Loading custom MSP library from QACs_51_STD_and_emerging_B.msp...

--- Processing: B1-10-MSMS-pos_I1.mzML ---

--- Processing: B1-10-MSMS-pos_I2.mzML ---

--- Processing: B1-10-MSMS-pos_I3.mzML ---

--- Processing: B1-10-MSMS-pos_I4.mzML ---

--- Processing: B1-10-MSMS-pos_I5.mzML ---

--- Processing: B1-10-MSMS-pos_I6.mzML ---

--- Processing: B1-10pool_AutoMSMS_pos_r1a.mzML ---

--- Processing: B1-10pool_AutoMSMS_pos_r2a.mzML ---

--- Processing: B1-10pool_TargMSMS_pos_r1a.mzML ---

--- Processing: B1-10pool_TargMSMS_pos_r2a.mzML ---

--- Processing: B11-20pool_MSMS_pos_r1.mzML ---
2026-04-16 08:36:05,607:WARNING:matchms:add_precursor_mz:No precursor_mz found in metadata.
2026-04-16 08:36:05,614:WARNING:matchms:add_precursor_mz:No precursor_mz found in metadata.
  [MATCH] DTXSID20975463 | Score: 0.000 | Peaks: 0
2026-04-16 08:36:05,924:WARNING:matchms:add_precursor_mz:No precursor_mz found in metadata.
2026-04-16 08:36:05,926:WARNING:matchms:add_precursor

In [ ]:
#isotopic matching


In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

# Replace with your full, un-truncated SMILES strings
smiles_list = [
    "CCCCCCCCCCCC[N+](C)(C)C",
    "CCCCCCCCCCCC[N+](C)(C)CC1=CC=CC=C1",
    "C[N+](C)(C)CCO",
    "C[N+](C)(C)CCCC([O-])=O",
    "CC[N+](CC)(CC)CC"
    # Add the rest of your full SMILES here
]

# Convert SMILES to RDKit molecule objects
molecules = [Chem.MolFromSmiles(smiles) for smiles in smiles_list]

# Keep the bold and large formatting
draw_opts = Draw.MolDrawOptions()
draw_opts.bondLineWidth = 4.0  
draw_opts.minFontSize = 24     
draw_opts.padding = 0.01       

# Loop through each molecule one by one
for i, mol in enumerate(molecules):
    if mol is not None:
        # Generate a single image for the current molecule
        img = Draw.MolToImage(mol, size=(300, 300), options=draw_opts)
        
        # Save each one with a unique filename (e.g., molecule_1.png, molecule_2.png)
        filename = f"molecule_{i+1}.png"
        img.save(filename)
        
        # Print the SMILES string and display the image inline
        print(f"Molecule {i+1}: {smiles_list[i]}")
        display(img)
    else:
        print(f"Failed to parse SMILES at index {i}: {smiles_list[i]}")